
# Step 1b: Grad‑CAM Inspection Notebook

Use this notebook to **visualize what your classifier is focusing on** (e.g., human vs. avatar vs. animal).  
It supports:
- Loading a SavedModel (`.keras` or SavedModel dir) or a compiled `tf.keras` model
- Directory or CSV dataset indexing
- Generating Grad‑CAM heatmaps for **misclassified** and **correct** examples per class
- Saving side‑by‑side overlays to disk for reports
- (Optional) A simple **border‑attention score** that can hint at shortcut risks (logos/borders)



## 0) Requirements

```bash
pip install tensorflow pillow opencv-python-headless numpy pandas matplotlib tqdm scikit-learn
```



## 1) User Tunables


In [ ]:

from pathlib import Path

# ===== USER TUNABLES =====
MODEL_PATH = Path("models/baseline_savedmodel")  # dir or file (.keras / .h5 / SavedModel dir)
CLASS_NAMES = None

DATA_ROOT = Path("data") 
METADATA_CSV = None      

TARGET_SPLIT = "val"    
IMG_SIZE = 224          
BATCH = 32

PREPROCESS = "resnet50"  # 'resnet50' | 'efficientnet' | 'none'
TARGET_LAYER_NAME = None # e.g., "conv5_block3_out"; None => auto-detect last conv
ALPHA = 0.35             

N_MISCLASS_PER_CLASS = 8 
N_CORRECT_PER_CLASS = 6  

OUT_DIR = Path("gradcam_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
# =========================



## 2) Imports & GPU Check


In [ ]:

import os, json, math, random
import numpy as np
import pandas as pd
import tensorflow as tf
from typing import List, Tuple
from tqdm import tqdm

from PIL import Image
import matplotlib.pyplot as plt
import cv2

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))



## 3) Load Model


In [ ]:

def load_model_any(path: Path) -> tf.keras.Model:
    m = tf.keras.models.load_model(path, compile=False)
    try:
        m.compile()
    except Exception:
        pass
    return m

model = load_model_any(MODEL_PATH)
model.summary()



## 4) Build Dataset Index (Directory or CSV)


In [ ]:

def list_images_directory(root: Path, split: str) -> pd.DataFrame:
    rows = []
    base = root / split
    if not base.exists():
        return pd.DataFrame(columns=["path","label","split"])
    for cls_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for img in cls_dir.rglob("*"):
            if img.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}:
                rows.append({"path": str(img.as_posix()), "label": cls_dir.name, "split": split})
    return pd.DataFrame(rows)

if METADATA_CSV:
    df_all = pd.read_csv(METADATA_CSV)
    need = {"path","label","split"}
    if not need.issubset(set(df_all.columns)):
        raise ValueError(f"CSV must contain columns: {need}")
    df_all["path"] = df_all["path"].astype(str)
else:
    df_train = list_images_directory(DATA_ROOT, "train")
    df_val   = list_images_directory(DATA_ROOT, "val")
    df_test  = list_images_directory(DATA_ROOT, "test")
    df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)

df = df_all[df_all["split"] == TARGET_SPLIT].copy().reset_index(drop=True)

if CLASS_NAMES is None:
    CLASS_NAMES = sorted(df_all["label"].dropna().unique().tolist())

label_to_index = {c:i for i,c in enumerate(CLASS_NAMES)}
index_to_label = {i:c for c,i in label_to_index.items()}

print("Classes:", CLASS_NAMES)
print("Counts:", df["label"].value_counts())



## 5) tf.data Pipeline & Preprocessing


In [ ]:

from tensorflow.keras.applications import resnet50, efficientnet

def preprocess_image(path: tf.Tensor) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32)
    if PREPROCESS.lower() == "resnet50":
        img = resnet50.preprocess_input(img)
    elif PREPROCESS.lower() == "efficientnet":
        img = efficientnet.preprocess_input(img)
    else:
        img = img / 255.0
    return img

def build_ds(paths: List[str], labels: List[int], batch=BATCH, shuffle=False) -> tf.data.Dataset:
    x = tf.constant(paths)
    y = tf.constant(labels, dtype=tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((x,y))
    if shuffle:
        ds = ds.shuffle(len(paths), reshuffle_each_iteration=False)
    ds = ds.map(lambda p,l: (preprocess_image(p), tf.one_hot(l, depth=len(CLASS_NAMES))), 
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

paths = df["path"].tolist()
labels = [label_to_index[l] for l in df["label"].tolist()]
ds_eval = build_ds(paths, labels, batch=BATCH, shuffle=False)



## 6) Predict & Build a Score Table


In [ ]:

probs = []
for xb, yb in ds_eval:
    p = model.predict(xb, verbose=0)
    probs.append(p)
probs = np.vstack(probs)

pred_idx = probs.argmax(axis=1)
pred_lbl = [index_to_label[i] for i in pred_idx]
true_lbl = df["label"].tolist()

conf = probs[np.arange(len(probs)), pred_idx]

score_df = pd.DataFrame({
    "path": paths,
    "true_label": true_lbl,
    "pred_label": pred_lbl,
    "pred_idx": pred_idx,
    "true_idx": [label_to_index[t] for t in true_lbl],
    "confidence": conf
})
score_df["is_correct"] = score_df["true_label"] == score_df["pred_label"]

score_df.head()



## 7) Grad‑CAM Utilities


In [ ]:

def find_last_conv_layer(model: tf.keras.Model) -> str:
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")

last_conv_name = TARGET_LAYER_NAME or find_last_conv_layer(model)
last_conv = model.get_layer(last_conv_name)
print("Grad‑CAM target layer:", last_conv_name)

grad_model = tf.keras.models.Model(
    [model.inputs], [last_conv.output, model.output]
)

def make_gradcam_heatmap(img_tensor: tf.Tensor, class_index: int) -> np.ndarray:
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_tensor, training=False)
        if class_index is None:
            class_index = tf.argmax(preds[0])
        target = preds[:, class_index]

    grads = tape.gradient(target, conv_out)                      
    pooled_grads = tf.reduce_mean(grads, axis=(1,2))             
    conv_out = conv_out[0]                                       
    pooled_grads = pooled_grads[0]                               

    heatmap = tf.tensordot(conv_out, pooled_grads, axes=(2,0))  
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0, 0)
    overlay = overlay[:, :, ::-1]
    return img_np, overlay

def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b>0 and (h-2*b)>0 and (w-2*b)>0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)



## 8) Visualize Grad‑CAM Panels


In [ ]:

import math

def pick_samples(score_df: pd.DataFrame, per_class_mis:int, per_class_ok:int):
    rows = []
    for cls in CLASS_NAMES:
        sub = score_df[score_df["true_label"] == cls].copy()
        mis = sub[~sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_mis)
        ok  = sub[sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_ok)
        rows.append(("MIS", cls, mis))
        rows.append(("OK",  cls, ok))
    return rows

def preprocess_for_single(path: str) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32)
    if PREPROCESS.lower() == "resnet50":
        from tensorflow.keras.applications.resnet50 import preprocess_input
        img = preprocess_input(img)
    elif PREPROCESS.lower() == "efficientnet":
        from tensorflow.keras.applications.efficientnet import preprocess_input
        img = preprocess_input(img)
    else:
        img = img / 255.0
    return tf.expand_dims(img, 0)

def panel_for_group(kind: str, cls: str, group_df: pd.DataFrame, save_path: Path):
    n = len(group_df)
    if n == 0:
        return

    cols = 3
    rows = n
    fig_h = max(4, rows * 3)
    fig_w = 12

    plt.figure(figsize=(fig_w, fig_h))
    idx = 1
    records = []

    for r in group_df.itertuples(index=False):
        x = preprocess_for_single(r.path)
        heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
        img_np, overlay = overlay_heatmap_on_image(r.path, heat)
        frac = border_attention_fraction(heat)

        plt.subplot(n, cols, idx);   plt.imshow(img_np);  plt.axis("off"); 
        plt.title(f"Orig\ntrue={r.true_label}\npred={r.pred_label}\nconf={r.confidence:.2f}")
        idx += 1
        plt.subplot(n, cols, idx);   plt.imshow(heat, cmap="jet");  plt.axis("off");  plt.title("Heatmap")
        idx += 1
        plt.subplot(n, cols, idx);   plt.imshow(overlay); plt.axis("off"); 
        plt.title(f"Overlay\nborder={frac:.2f}")
        idx += 1

        records.append({
            "path": r.path,
            "true_label": r.true_label,
            "pred_label": r.pred_label,
            "confidence": r.confidence,
            "border_attention_frac": frac,
            "is_correct": r.is_correct
        })

    plt.suptitle(f"{kind}: {cls} — {n} samples", y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.show()

    pd.DataFrame(records).to_csv(save_path.with_suffix(".csv"), index=False)

samples = pick_samples(score_df, N_MISCLASS_PER_CLASS, N_CORRECT_PER_CLASS)
for kind, cls, gdf in samples:
    slug = f"{kind.lower()}_{cls}".replace(" ", "_")
    out_file = OUT_DIR / f"gradcam_{slug}.png"
    panel_for_group(kind, cls, gdf, out_file)

print("Saved panels to:", OUT_DIR.resolve())



## 9) Single‑Image Helper


In [ ]:

def gradcam_single_image(path: str, class_idx: int = None):
    x = preprocess_for_single(path)
    if class_idx is None:
        p = model.predict(x, verbose=0)[0]
        class_idx = int(np.argmax(p))
    heat = make_gradcam_heatmap(x, class_index=class_idx)
    img_np, overlay = overlay_heatmap_on_image(path, heat)
    frac = border_attention_fraction(heat)

    plt.figure(figsize=(10,3))
    plt.subplot(1,3,1); plt.imshow(img_np); plt.axis("off"); plt.title("Original")
    plt.subplot(1,3,2); plt.imshow(heat, cmap="jet"); plt.axis("off"); plt.title("Heatmap")
    plt.subplot(1,3,3); plt.imshow(overlay); plt.axis("off"); plt.title(f"Overlay\nborder={frac:.2f}")
    plt.tight_layout()
    plt.show()

# Example:
# gradcam_single_image(df.iloc[0]['path'])



## 10) What to Look For

- **Correct**: heat concentrated on salient object (face/body/animal).
- **Wrong**: heat on **backgrounds, borders, logos, corner watermarks**, or text.
- If border‑attention fractions are systematically high, consider **masking/cropping** or **augmentations** that randomize edges.
